# Lecture 8 — Class Exercise
## Choropleth Maps

> **Push to:** `week08/lecture08_exercise.ipynb`

**Rules:**
1. Use `px.choropleth` or `px.choropleth_map` — choose deliberately and state your reason
2. Right colour scale for your data (sequential vs diverging) — state which and why
3. Insight title names a geographic finding — not just a topic
4. `featureidkey` must be correctly matched to your GeoJSON

---


In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import json


## Task 1 — World choropleth: life expectancy diverging scale

**What to build:** A world choropleth showing **life expectancy relative to the global average** using a diverging colour scale.

**Requirements:**
- Use the Gapminder dataset for 2007: `px.data.gapminder()`
- Compute each country's deviation from the global mean life expectancy
- Diverging scale centred at zero (= world average)
- `hover_data` showing country name, raw life expectancy, and deviation
- Insight title naming which region is furthest below average

> 💡 `gm_2007['lifeExp'].mean()` gives you the global average to subtract from


In [ ]:
# Task 1
# Using px.choropleth (not choropleth_map) because this is a full static world
# view meant for global comparison — no need for pan/zoom basemap tiles.
gm_2007 = px.data.gapminder().query("year == 2007").copy()

global_avg = gm_2007["lifeExp"].mean()
gm_2007["deviation"] = gm_2007["lifeExp"] - global_avg

# Continent furthest below the global average -> drives the insight title
lowest_continent = gm_2007.groupby("continent")["deviation"].mean().idxmin()

fig1 = px.choropleth(
    gm_2007,
    locations="iso_alpha",
    color="deviation",
    hover_name="country",
    hover_data={
        "lifeExp": ":.1f",
        "deviation": ":.1f",
        "iso_alpha": False,
    },
    # Diverging scale centred at 0 because the data is a signed deviation
    # from an average — magnitude AND direction (above/below) both matter.
    color_continuous_scale="RdBu",
    color_continuous_midpoint=0,
    title=f"Life Expectancy vs. Global Average (2007): {lowest_continent} Lags Furthest Below the World Mean",
)

fig1.update_layout(
    coloraxis_colorbar=dict(title="Deviation<br>(years)"),
    margin=dict(l=0, r=0, t=60, b=0),
)

fig1.show()


## Task 2 — Find your own GeoJSON

**What to build:** A choropleth using a GeoJSON file you find yourself online.

**Requirements:**
- Find a free GeoJSON file for any geography that interests you (country, region, city)
- Create or find a matching dataset with at least one numeric variable per region
- Build either a `px.choropleth` or `px.choropleth_mapbox` — state your choice and reason in the markdown cell below
- Correctly identify and set `featureidkey` by inspecting the GeoJSON properties
- Choose sequential or diverging scale — state your reason in the markdown cell below
- Insight title naming a geographic finding

**Where to find GeoJSON files:**
- [geojson.xyz](https://geojson.xyz/) — countries, cities, natural features
- [naturalearthdata.com](https://www.naturalearthdata.com/) — global admin boundaries
- [github.com/datasets/geo-countries](https://github.com/datasets/geo-countries) — country polygons
- Search: `[country name] [admin level] GeoJSON github` — most countries have free boundary files on GitHub

> 💡 Before plotting, always inspect your GeoJSON properties first:
> ```python
> print(my_geojson['features'][0]['properties'])
> ```
> The property name that matches your dataframe's location column is what goes in `featureidkey='properties.???'`


### Task 2 — Design decisions

**GeoJSON source:** US state boundaries from the PublicaMundi MappingAPI repo on GitHub —
`https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json`

**Chart type chosen** (`px.choropleth` or `px.choropleth_mapbox`) **and reason:**

`px.choropleth` with `scope="usa"`. A static, non-interactive-basemap view is enough here since
the goal is a single comparative snapshot across states, not zooming into local detail — that's
what `choropleth_mapbox` / `choropleth_map` are better suited for.

**Colour scale chosen** (sequential or diverging) **and reason:**

Sequential (`Viridis`). Population density is a magnitude that starts at zero with no natural
midpoint to diverge around, so a sequential scale (light = low, dark = high) is the right choice.


In [ ]:
# Task 2
import urllib.request

# Load the GeoJSON
geojson_url = "https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json"
with urllib.request.urlopen(geojson_url) as response:
    us_states_geojson = json.load(response)

# Inspect the properties to find the right featureidkey
print(us_states_geojson["features"][0]["properties"])
# -> {'name': 'Alabama', 'density': ..., ...}  so featureidkey="properties.name"

# Sample dataset: approximate population density (people per sq. mile).
# Illustrative values for a class exercise — swap in your own found dataset if you like.
density_data = {
    "California": 253.7, "Texas": 112.8, "Florida": 410.3, "New York": 412.5,
    "Pennsylvania": 286.8, "Illinois": 227.7, "Ohio": 288.7, "Georgia": 187.5,
    "North Carolina": 218.6, "Michigan": 177.5, "New Jersey": 1215.8,
    "Virginia": 218.4, "Washington": 118.1, "Arizona": 65.0, "Massachusetts": 894.4,
    "Tennessee": 174.7, "Indiana": 189.3, "Missouri": 89.4, "Maryland": 636.8,
    "Wisconsin": 108.8, "Colorado": 56.4, "Minnesota": 71.7, "South Carolina": 174.5,
    "Alabama": 99.9, "Louisiana": 107.5, "Kentucky": 115.6, "Oregon": 44.7,
    "Oklahoma": 57.7, "Connecticut": 736.4, "Utah": 40.5, "Iowa": 57.0,
    "Nevada": 28.3, "Arkansas": 58.7, "Mississippi": 63.5, "Kansas": 36.0,
    "New Mexico": 17.4, "Nebraska": 25.5, "West Virginia": 73.6, "Idaho": 22.9,
    "Hawaii": 226.2, "New Hampshire": 153.9, "Maine": 43.9, "Montana": 7.4,
    "Rhode Island": 1021.4, "Delaware": 508.4, "South Dakota": 11.7,
    "North Dakota": 11.2, "Alaska": 1.3, "Vermont": 68.1, "Wyoming": 6.0,
}
df_states = pd.DataFrame(list(density_data.items()), columns=["state", "density"])

fig2 = px.choropleth(
    df_states,
    geojson=us_states_geojson,
    locations="state",
    featureidkey="properties.name",
    color="density",
    scope="usa",
    color_continuous_scale="Viridis",
    hover_name="state",
    hover_data={"density": ":.1f"},
    title="Population Density: The Northeast Corridor Dwarfs the Mountain West",
)

fig2.update_layout(margin=dict(l=0, r=0, t=60, b=0))
fig2.show()
